# Audio Transcription Pipeline

This notebook:
1. Decodes base64 audio from JSON files and saves them as audio files
2. Transcribes audio files using Whisper (openai/whisper-large-v3-turbo)
3. Skips already processed files to enable incremental processing
4. Tracks progress and generates summary reports

## 1. Load Required Modules

In [1]:
import os
import json
import base64
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple
import mimetypes
from tqdm import tqdm
import traceback

# Whisper imports
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

print("✓ All modules imported successfully")

✓ All modules imported successfully


## 2. Configure Paths

In [2]:
# Set up directory paths
BASE_DIR = Path(os.getcwd())
DATA_DIR = BASE_DIR / "data" / "results_audio"
AUDIO_OUTPUT_DIR = BASE_DIR / "outputs" / "audio_files"
TRANSCRIPT_OUTPUT_DIR = BASE_DIR / "outputs" / "audio_transcripts"
LOGS_DIR = BASE_DIR / "outputs"

# Create output directories if they don't exist
AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Audio Output Directory: {AUDIO_OUTPUT_DIR}")
print(f"Transcript Output Directory: {TRANSCRIPT_OUTPUT_DIR}")
print(f"\n✓ Directories configured and created")

Working Directory: /Volumes/storage/PROJECTS/Article_ExperimentalUtopianPrototypes/Article_ExperimentalUtopianPrototypes/Analyses/mainStudy/01_dataPreperation
Data Directory: /Volumes/storage/PROJECTS/Article_ExperimentalUtopianPrototypes/Article_ExperimentalUtopianPrototypes/Analyses/mainStudy/01_dataPreperation/data/results_audio
Audio Output Directory: /Volumes/storage/PROJECTS/Article_ExperimentalUtopianPrototypes/Article_ExperimentalUtopianPrototypes/Analyses/mainStudy/01_dataPreperation/outputs/audio_files
Transcript Output Directory: /Volumes/storage/PROJECTS/Article_ExperimentalUtopianPrototypes/Article_ExperimentalUtopianPrototypes/Analyses/mainStudy/01_dataPreperation/outputs/audio_transcripts

✓ Directories configured and created


## 3. Define Helper Functions

In [3]:
def get_file_extension(mime_type: str) -> str:
    """
    Get file extension from MIME type.
    
    Args:
        mime_type: MIME type string (e.g., 'audio/wav', 'audio/webm; codecs=opus')
    
    Returns:
        File extension (e.g., '.wav', '.webm')
    """
    # Strip codec parameters (e.g., "audio/webm; codecs=opus" -> "audio/webm")
    base_mime = mime_type.split(';')[0].strip()
    
    mime_map = {
        'audio/wav': '.wav',
        'audio/mpeg': '.mp3',
        'audio/mp4': '.m4a',
        'audio/ogg': '.ogg',
        'audio/webm': '.webm',
        'audio/aac': '.aac',
    }
    return mime_map.get(base_mime, '.wav')


def generate_output_filename(json_data: Dict, json_filename: str) -> str:
    """
    Generate output filename for audio file.
    
    Args:
        json_data: Parsed JSON data containing audio metadata
        json_filename: Original JSON filename
    
    Returns:
        Filename for the audio file
    """
    id_person = json_data.get('id_person', 'unknown')
    audio_type = json_data.get('type', 'unknown')
    
    # Extract timestamp from filename if available
    parts = json_filename.replace('.json', '').split('_')
    timestamp = parts[-1] if len(parts) > 0 else datetime.now().strftime('%Y%m%d%H%M%S')
    
    ext = get_file_extension(json_data.get('audio_mime', 'audio/wav'))
    
    return f"{id_person}_{audio_type}_{timestamp}{ext}"


def decode_base64_audio(audio_data: str, output_path: Path) -> bool:
    """
    Decode base64 encoded audio and save to file.
    
    Args:
        audio_data: Base64 encoded audio string (may include data URI prefix)
        output_path: Path to save the audio file
    
    Returns:
        True if successful, False otherwise
    """
    try:
        # Strip data URI prefix if present
        if audio_data.startswith('data:'):
            # Extract the base64 part after "base64,"
            audio_data = audio_data.split('base64,', 1)[1]
        
        audio_bytes = base64.b64decode(audio_data)
        with open(output_path, 'wb') as f:
            f.write(audio_bytes)
        return True
    except Exception as e:
        print(f"Error decoding base64: {str(e)}")
        return False


def get_existing_files(directory: Path) -> set:
    """
    Get set of existing files in directory.
    
    Args:
        directory: Path to directory
    
    Returns:
        Set of filenames
    """
    if directory.exists():
        return set(f.name for f in directory.iterdir() if f.is_file())
    return set()


print("✓ Helper functions defined")

✓ Helper functions defined


## 4. Scan for Input JSON Files

In [4]:
def find_audio_json_files(root_dir: Path) -> List[Tuple[Path, str]]:
    """
    Recursively find all JSON files containing audio data.
    
    Args:
        root_dir: Root directory to search
    
    Returns:
        List of tuples (file_path, relative_path)
    """
    json_files = []
    
    for json_file in root_dir.rglob('*.json'):
        # Check if file is in a 'files' subdirectory
        if 'files' in json_file.parts:
            relative_path = '/'.join(json_file.parts[-3:])  # study_result/comp-result/files/filename
            json_files.append((json_file, relative_path))
    
    return sorted(json_files)


# Find all JSON files with audio data
json_files = find_audio_json_files(DATA_DIR)
print(f"Found {len(json_files)} JSON files with audio data:")
for i, (file_path, rel_path) in enumerate(json_files[:5]):
    print(f"  {i+1}. {rel_path}")
if len(json_files) > 5:
    print(f"  ... and {len(json_files) - 5} more")

Found 10 JSON files with audio data:
  1. comp-result_26881/files/audio_missing_utopia_bbbb_1779173663003.json
  2. comp-result_26881/files/audio_ranking_explanation_bbbb_1779173647432.json
  3. comp-result_26882/files/audio_missing_utopia_cc33_1779175920347.json
  4. comp-result_26882/files/audio_ranking_explanation_cc33_1779175914973.json
  5. comp-result_26883/files/audio_missing_utopia_111_1779276727208.json
  ... and 5 more


## 5. Step 1: Decode Base64 Audio Files

In [5]:
def decode_all_audio_files(json_files: List[Tuple[Path, str]], 
                           output_dir: Path,
                           existing_files: set) -> Dict:
    """
    Decode all base64 audio files from JSON.
    
    Args:
        json_files: List of JSON file paths
        output_dir: Directory to save decoded audio files
        existing_files: Set of existing files to skip
    
    Returns:
        Dictionary with processing results
    """
    results = {
        'processed': [],
        'skipped': [],
        'errors': [],
        'total_decoded': 0,
        'total_skipped': 0,
    }
    
    print("\n" + "="*80)
    print("STEP 1: DECODING BASE64 AUDIO FILES")
    print("="*80)
    
    for json_path, rel_path in tqdm(json_files, desc="Decoding audio"):
        try:
            with open(json_path, 'r') as f:
                json_data = json.load(f)
            
            # Generate output filename
            output_filename = generate_output_filename(json_data, json_path.name)
            output_path = output_dir / output_filename
            
            # Check if file already exists
            if output_filename in existing_files:
                results['skipped'].append({
                    'json_file': rel_path,
                    'audio_file': output_filename,
                    'reason': 'File already exists'
                })
                results['total_skipped'] += 1
                continue
            
            # Decode and save audio
            if 'audio' in json_data:
                success = decode_base64_audio(json_data['audio'], output_path)
                
                if success:
                    results['processed'].append({
                        'json_file': rel_path,
                        'audio_file': output_filename,
                        'id_person': json_data.get('id_person'),
                        'type': json_data.get('type'),
                        'audio_duration_seconds': json_data.get('audio_duration_seconds'),
                        'audio_size_bytes': json_data.get('audio_size_bytes'),
                    })
                    results['total_decoded'] += 1
                else:
                    results['errors'].append({
                        'json_file': rel_path,
                        'error': 'Failed to decode base64'
                    })
            else:
                results['errors'].append({
                    'json_file': rel_path,
                    'error': 'No audio field in JSON'
                })
        
        except Exception as e:
            results['errors'].append({
                'json_file': rel_path,
                'error': f"{type(e).__name__}: {str(e)}"
            })
    
    return results





# Get existing audio files
existing_audio_files = get_existing_files(AUDIO_OUTPUT_DIR)
print(f"Found {len(existing_audio_files)} existing audio files (will be skipped)")

# Decode all audio files
decode_results = decode_all_audio_files(json_files, AUDIO_OUTPUT_DIR, existing_audio_files)

print(f"\n✓ Audio Decoding Summary:")
print(f"  - Total processed: {decode_results['total_decoded']}")
print(f"  - Total skipped: {decode_results['total_skipped']}")
print(f"  - Total errors: {len(decode_results['errors'])}")

Found 10 existing audio files (will be skipped)

STEP 1: DECODING BASE64 AUDIO FILES


Decoding audio: 100%|██████████| 10/10 [00:00<00:00, 627.59it/s]


✓ Audio Decoding Summary:
  - Total processed: 0
  - Total skipped: 10
  - Total errors: 0


## 6. Step 2: Initialize Whisper Model

In [6]:
print("\n" + "="*80)
print("STEP 2: INITIALIZING WHISPER MODEL")
print("="*80)

# Check device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Device: {device}")
print(f"Data type: {dtype}")

# Load Whisper model
print("\nLoading Whisper large-v3-turbo model...")
model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

# Create pipeline with return_timestamps=True for better long-form transcription
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=dtype,
    device=device,
    return_timestamps=True,  # ✅ IMPORTANT: Add this here
)

print("✓ Whisper model loaded successfully")


STEP 2: INITIALIZING WHISPER MODEL
Device: cpu
Data type: torch.float32

Loading Whisper large-v3-turbo model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✓ Whisper model loaded successfully


## 7. Step 3: Transcribe Audio Files

In [7]:
import av
import soundfile as sf
import numpy as np
import librosa
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import json
import shutil
from typing import Dict, List, Tuple

# Remove/replace the old ffmpeg function with this:
def convert_webm_to_wav_pyav(input_path: Path, output_path: Path) -> bool:
    """
    Convert WebM audio to WAV using PyAV (uses libavcodec library).
    """
    try:
        # Open WebM file
        container = av.open(str(input_path))
        audio_stream = None
        
        # Find audio stream
        for stream in container.streams:
            if stream.type == 'audio':
                audio_stream = stream
                break
        
        if not audio_stream:
            print(f"  No audio stream found in {input_path.name}")
            return False
        
        # Decode audio
        audio_data = []
        sample_rate = audio_stream.sample_rate
        
        for frame in container.decode(audio_stream):
            audio_data.append(frame.to_ndarray())
        
        # Concatenate and convert to mono
        if audio_data:
            audio = np.concatenate(audio_data, axis=1)
            if audio.shape[0] > 1:  # Convert stereo to mono
                audio = audio.mean(axis=0)
            else:
                audio = audio[0]
            
            # Resample to 16kHz
            if sample_rate != 16000:
                audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=16000)
                sample_rate = 16000
            
            # Normalize
            audio = audio / np.max(np.abs(audio))
            
            # Save as WAV
            sf.write(str(output_path), audio, sample_rate)
            return True
        
        return False
        
    except Exception as e:
        print(f"  Error converting WebM with PyAV: {str(e)}")
        return False



def transcribe_audio_files(audio_dir: Path,
                          transcript_dir: Path,
                          pipe,
                          existing_transcripts: set) -> Dict:
    """
    Transcribe audio files using Whisper.
    """
    results = {
        'processed': [],
        'skipped': [],
        'errors': [],
        'total_transcribed': 0,
        'total_skipped': 0,
    }
    
    print("\n" + "="*80)
    print("STEP 3: TRANSCRIBING AUDIO FILES WITH WHISPER")
    print("="*80)
    
    # Create temp directory for WAV conversion
    dir_audio_wav = audio_dir.parent / 'audio_files_wav'
    dir_audio_wav.mkdir(exist_ok=True)
    
    # Get all audio files
    audio_files = sorted([f for f in audio_dir.iterdir() if f.is_file()])
    print(f"Found {len(audio_files)} audio files to process")
    
    for audio_file in tqdm(audio_files, desc="Transcribing audio"):
        transcript_filename = audio_file.stem + '_transcript.json'
        
        try:
            transcript_path = transcript_dir / transcript_filename
            
            # Check if transcript already exists
            if transcript_filename in existing_transcripts:
                results['skipped'].append({
                    'audio_file': audio_file.name,
                    'transcript_file': transcript_filename,
                    'reason': 'Transcript already exists'
                })
                results['total_skipped'] += 1
                continue
            
            # Convert WebM to WAV if needed
            wav_path = audio_file
            if audio_file.suffix.lower() == '.webm':
                wav_path = dir_audio_wav / (audio_file.stem + '.wav')
                
                if not wav_path.exists():
                    print(f"\n  Converting {audio_file.name} to WAV...")
                    if not convert_webm_to_wav_pyav(audio_file, wav_path):
                        results['errors'].append({
                            'audio_file': audio_file.name,
                            'error': 'Failed to convert WebM to WAV'
                        })
                        continue
            
            # Load audio directly with soundfile (no ffmpeg needed)
            start_time = datetime.now()
            try:
                audio_data, sample_rate = sf.read(str(wav_path))
                
                # Resample to 16kHz if needed
                if sample_rate != 16000:
                    audio_data = librosa.resample(audio_data, orig_sr=sample_rate, target_sr=16000)
                
                # Pass numpy array directly to pipeline
                # Remove sample_rate parameter and batch_size - use defaults
                result = pipe(audio_data)
                
                processing_time = (datetime.now() - start_time).total_seconds()
                
                # Extract text - handle both timestamp and non-timestamp formats
                if 'chunks' in result:
                    # When return_timestamps=True, text is in chunks
                    transcript_text = ''.join([chunk['text'] for chunk in result['chunks']])
                else:
                    # Standard format
                    transcript_text = result.get('text', '')
                
                # Prepare transcript data
                transcript_data = {
                    'audio_file': audio_file.name,
                    'transcript': transcript_text,
                    'language': result.get('language', 'unknown'),
                    'processing_time_seconds': processing_time,
                    'timestamp': datetime.now().isoformat(),
                    'model': 'openai/whisper-large-v3-turbo',
                }
                
                # Save full result if timestamps were returned
                if 'chunks' in result:
                    transcript_data['chunks'] = result['chunks']
                
                # Save transcript
                with open(transcript_path, 'w') as f:
                    json.dump(transcript_data, f, indent=2)
                
                results['processed'].append({
                    'audio_file': audio_file.name,
                    'transcript_file': transcript_filename,
                    'transcript_length': len(transcript_text),
                    'processing_time_seconds': processing_time,
                })
                results['total_transcribed'] += 1
                
            except Exception as pipe_error:
                error_msg = f"{type(pipe_error).__name__}: {str(pipe_error)}"
                print(f"\n  ✗ Error transcribing {audio_file.name}:")
                print(f"    {error_msg}")
                results['errors'].append({
                    'audio_file': audio_file.name,
                    'error': error_msg
                })
        
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            results['errors'].append({
                'audio_file': audio_file.name,
                'error': error_msg
            })
    
    # Clean up temp directory
    # if dir_audio_wav.exists():
    #    shutil.rmtree(dir_audio_wav)
    #    print(f"\nCleaned up WAV files")
    
    return results





# Run transcription
existing_transcripts = get_existing_files(TRANSCRIPT_OUTPUT_DIR)
print(f"Found {len(existing_transcripts)} existing transcripts (will be skipped)")

transcribe_results = transcribe_audio_files(
    AUDIO_OUTPUT_DIR,
    TRANSCRIPT_OUTPUT_DIR,
    pipe,
    existing_transcripts
)

print(f"\n✓ Transcription Summary:")
print(f"  - Total transcribed: {transcribe_results['total_transcribed']}")
print(f"  - Total skipped: {transcribe_results['total_skipped']}")
print(f"  - Total errors: {len(transcribe_results['errors'])}")

if transcribe_results['errors']:
    print(f"\nError Details:")
    for error in transcribe_results['errors']:
        print(f"  - {error['audio_file']}: {error['error']}")

Found 0 existing transcripts (will be skipped)

STEP 3: TRANSCRIBING AUDIO FILES WITH WHISPER
Found 10 audio files to process


Transcribing audio:   0%|          | 0/10 [00:00<?, ?it/s]


  Converting 111_audio_missing_utopia_1779276727208.webm to WAV...


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr


  Converting 111_audio_missing_utopia_1779750864225.webm to WAV...


...


  Converting cc33_audio_missing_utopia_1779175920347.webm to WAV...


Transcribing audio:  90%|█████████ | 9/10 [01:06<00:05,  5.95s/it]


  Converting cc33_audio_ranking_explanation_1779175914973.webm to WAV...


Transcribing audio: 100%|██████████| 10/10 [01:10<00:00,  7.10s/it]


✓ Transcription Summary:
  - Total transcribed: 10
  - Total skipped: 0
  - Total errors: 0


## 8. Step 4: Generate Summary Reports

In [10]:
print("\n" + "="*80)
print("STEP 4: GENERATING SUMMARY REPORTS")
print("="*80)

# Create summary DataFrame for decoded audio
if decode_results['processed']:
    decode_df = pd.DataFrame(decode_results['processed'])
    decode_csv_path = LOGS_DIR / f"decode_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    decode_df.to_csv(decode_csv_path, index=False)
    print(f"\n✓ Audio decode summary saved to: {decode_csv_path.name}")
    print(f"  - {len(decode_df)} files decoded")

# Create summary DataFrame for transcriptions
if transcribe_results['processed']:
    transcribe_df = pd.DataFrame(transcribe_results['processed'])
    transcribe_csv_path = LOGS_DIR / f"transcription_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    transcribe_df.to_csv(transcribe_csv_path, index=False)
    print(f"\n✓ Transcription summary saved to: {transcribe_csv_path.name}")
    print(f"  - {len(transcribe_df)} files transcribed")
    
    # Show average processing time
    avg_time = transcribe_df['processing_time_seconds'].mean()
    print(f"  - Average processing time: {avg_time:.2f} seconds")

# Save error logs if there are any
all_errors = decode_results['errors'] + transcribe_results['errors']
if all_errors:
    error_df = pd.DataFrame(all_errors)
    error_csv_path = LOGS_DIR / f"error_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    error_df.to_csv(error_csv_path, index=False)
    print(f"\n⚠ Error log saved to: {error_csv_path.name}")
    print(f"  - {len(error_df)} errors encountered")


STEP 4: GENERATING SUMMARY REPORTS

✓ Transcription summary saved to: transcription_summary_20260530_130410.csv
  - 10 files transcribed
  - Average processing time: 6.96 seconds


## 9. Final Summary

In [11]:
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\n📊 DECODING RESULTS:")
print(f"  ✓ Decoded: {decode_results['total_decoded']}")
print(f"  ⊘ Skipped: {decode_results['total_skipped']}")
print(f"  ✗ Errors: {len(decode_results['errors'])}")

print(f"\n📊 TRANSCRIPTION RESULTS:")
print(f"  ✓ Transcribed: {transcribe_results['total_transcribed']}")
print(f"  ⊘ Skipped: {transcribe_results['total_skipped']}")
print(f"  ✗ Errors: {len(transcribe_results['errors'])}")

print(f"\n📁 OUTPUT LOCATIONS:")
print(f"  Audio files: {AUDIO_OUTPUT_DIR}")
print(f"  Transcripts: {TRANSCRIPT_OUTPUT_DIR}")
print(f"  Logs: {LOGS_DIR}")

total_processed = decode_results['total_decoded'] + transcribe_results['total_transcribed']
total_skipped = decode_results['total_skipped'] + transcribe_results['total_skipped']
print(f"\n✓ Pipeline completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Total items processed: {total_processed}")
print(f"  Total items skipped: {total_skipped}")


FINAL SUMMARY

📊 DECODING RESULTS:
  ✓ Decoded: 0
  ⊘ Skipped: 10
  ✗ Errors: 0

📊 TRANSCRIPTION RESULTS:
  ✓ Transcribed: 10
  ⊘ Skipped: 0
  ✗ Errors: 0

📁 OUTPUT LOCATIONS:
  Audio files: /Volumes/storage/PROJECTS/Article_ExperimentalUtopianPrototypes/Article_ExperimentalUtopianPrototypes/Analyses/mainStudy/01_dataPreperation/outputs/audio_files
  Transcripts: /Volumes/storage/PROJECTS/Article_ExperimentalUtopianPrototypes/Article_ExperimentalUtopianPrototypes/Analyses/mainStudy/01_dataPreperation/outputs/audio_transcripts
  Logs: /Volumes/storage/PROJECTS/Article_ExperimentalUtopianPrototypes/Article_ExperimentalUtopianPrototypes/Analyses/mainStudy/01_dataPreperation/outputs

✓ Pipeline completed at 2026-05-30 13:04:13
  Total items processed: 10
  Total items skipped: 10
